# 04 - Urdu zero-shot check on the final model (v1)

**Question:** our model was trained on English only. XLM-RoBERTa was pre-trained on 100 languages including Urdu, so it may already recognise some Urdu entities. How many?

**No training, no changes to the model.** We run v1 (`xlmr_base/best`) on 8 Urdu sentences (plus one long Urdu paragraph) and compare with the entities *we* expect. This is a quick sanity check on a tiny sample, NOT a benchmark: use it only to decide what to tell the organisers and what to say in the presentation.

Runtime > Change runtime type > **T4 GPU**, then run the cells top to bottom.

## 1. GPU check and install

In [ ]:
!nvidia-smi
!pip -q install -U transformers accelerate

## 2. Upload `colab_bundle.zip` (only the scripts are needed) and unzip
The zip already has a top-level `ner/` folder.

In [ ]:
from google.colab import files
up = files.upload()
!rm -rf ner
!unzip -oq colab_bundle.zip
%cd ner
!ls scripts

## 3. Copy ONLY the v1 model from Drive (about 1.1 GB)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p outputs/xlmr_base
!cp -r /content/drive/MyDrive/ner_results/xlmr_base/best outputs/xlmr_base/
!ls outputs/xlmr_base/best

## 4. Urdu test sentences with the entities we expect
Each sentence lists `(text, label)` pairs that a careful human would mark using our guidelines. Written by hand, so they are small and only roughly validated. All phone and CNIC numbers are fake.

The last cell of this section also joins all sentences into one long Urdu paragraph.

In [ ]:
import json
URDU = [
  ("وزیراعظم شہباز شریف نے لاہور میں پاکستان تحریک انصاف کے رہنما عمران خان سے ملاقات کی۔",
   [("شہباز شریف","PERSON"),("لاہور","CITY"),("پاکستان تحریک انصاف","POLITICAL_PARTY"),("عمران خان","PERSON")]),
  ("سپریم کورٹ آف پاکستان نے منگل کو انتخابی ایکٹ 2017 کے خلاف درخواست پر سماعت کی۔",
   [("سپریم کورٹ آف پاکستان","COURT_LEGAL_BODY"),("منگل","DATE"),("انتخابی ایکٹ 2017","LAW_ACT")]),
  ("ڈان اور جیو نیوز کے مطابق حکومت نے سیلاب متاثرین کے لیے 25 ارب روپے مختص کیے۔",
   [("ڈان","MEDIA_OUTLET"),("جیو نیوز","MEDIA_OUTLET"),("25 ارب روپے","MONEY")]),
  ("کراچی یونیورسٹی کے طلبہ نے 2 اکتوبر کو شام 4 بجے کراچی پریس کلب کے باہر احتجاج کیا۔",
   [("کراچی یونیورسٹی","EDUCATIONAL_INSTITUTION"),("2 اکتوبر","DATE"),("شام 4 بجے","TIME"),("کراچی پریس کلب","LANDMARK_FACILITY")]),
  ("پاکستان نے ایشیا کپ میں بھارت کو شکست دی اور بابر اعظم نے سنچری بنائی۔",
   [("پاکستان","SPORTS_TEAM"),("ایشیا کپ","EVENT"),("بھارت","SPORTS_TEAM"),("بابر اعظم","PERSON")]),
  ("کوئٹہ، پشاور اور اسلام آباد میں مہنگائی 15 فیصد بڑھ گئی۔",
   [("کوئٹہ","CITY"),("پشاور","CITY"),("اسلام آباد","CITY"),("15 فیصد","PERCENT")]),
  ("رابطے کے لیے 0300-1234567 پر کال کریں اور شناختی کارڈ نمبر 35202-1234567-1 بتائیں۔",
   [("0300-1234567","PHONE_NUMBER"),("35202-1234567-1","ID_NUMBER")]),
  ("الیکشن کمیشن آف پاکستان نے خیبر پختونخوا اور بلوچستان میں انتخابات کا اعلان کیا۔",
   [("الیکشن کمیشن آف پاکستان","GOVERNMENT_BODY"),("خیبر پختونخوا","PROVINCE_REGION"),("بلوچستان","PROVINCE_REGION")]),
]
with open('urdu_sentences.txt', 'w', encoding='utf-8') as f:
    for s, _ in URDU: f.write(s + '\n')
# one long paragraph (one line) from all the sentences
with open('urdu_paragraph.txt', 'w', encoding='utf-8') as f:
    f.write(' '.join(s for s, _ in URDU) + '\n')
print(len(URDU), 'sentences saved;', sum(len(e) for _, e in URDU), 'expected entities')

## 5. Run v1 on the Urdu sentences
Prints what the model found in each sentence.

In [ ]:
!python scripts/predict.py --model outputs/xlmr_base/best --input urdu_sentences.txt --out urdu_preds.jsonl 2>&1 | grep -v "Loading weights"

## 6. Compare with what we expected
For every expected entity: **OK** = same text and same label found, **LABEL** = right words but wrong label, **MISS** = nothing found there. **EXTRA** = the model found something we did not expect (may be wrong, or may be a valid entity we forgot).
The final lines give recall (expected entities found with the right label), precision (found entities that were expected) and F1, with the same exact-match rule as the English scoring.

In [ ]:
preds = [json.loads(l) for l in open('urdu_preds.jsonl', encoding='utf-8')]
tp = fp = fn = 0
by_label = {}
for (sent, exp), p in zip(URDU, preds):
    got = [(e['text'], e['label']) for e in p['entities']]
    gotd = {}
    for t, l in got: gotd.setdefault(t, []).append(l)
    print('\n' + sent)
    for t, l in exp:
        d = by_label.setdefault(l, [0, 0]); d[1] += 1
        if (t, l) in got: print(f'   OK     {l:24} {t}'); tp += 1; d[0] += 1
        elif t in gotd: print(f'   LABEL  {l:24} {t}   (model said {gotd[t]})'); fn += 1
        else: print(f'   MISS   {l:24} {t}'); fn += 1
    for t, l in got:
        if (t, l) not in exp: print(f'   EXTRA  {l:24} {t}'); fp += 1
P = tp / (tp + fp) if tp + fp else 0; R = tp / (tp + fn) if tp + fn else 0
F = 2 * P * R / (P + R) if P + R else 0
print(f'\nURDU zero-shot, v1:  found right {tp} of {tp + fn} expected  |  precision {P:.2f}  recall {R:.2f}  F1 {F:.2f}')
print('\nrecall per label (right / expected):')
for l, (a, b) in sorted(by_label.items()): print(f'   {l:24} {a}/{b}')

## 7. One long Urdu paragraph
All 8 sentences joined into one line. `predict.py` cuts long text into 100-word windows. Check whether results stay similar when the sentences are not separate.

In [ ]:
!python scripts/predict.py --model outputs/xlmr_base/best --input urdu_paragraph.txt --out urdu_paragraph_preds.jsonl 2>&1 | grep -v "Loading weights"

## 8. Your own Urdu text (optional)
Type or paste any Urdu text and see what v1 finds.

In [ ]:
MY_TEXT = "وزیر خزانہ نے اسلام آباد میں آئی ایم ایف کے وفد سے ملاقات کی۔"
!python scripts/predict.py --model outputs/xlmr_base/best --text "{MY_TEXT}" 2>&1 | grep -v "Loading weights"

## 9. Download the results
Put the zip in `trained_reports/urdu_check/` on your PC and tell me.

In [ ]:
!zip -q urdu_check.zip urdu_sentences.txt urdu_paragraph.txt urdu_preds.jsonl urdu_paragraph_preds.jsonl
from google.colab import files
files.download('urdu_check.zip')